# ⚡ EnergyPlus Simulation: VS Code + GCP Compute Engine VM

This notebook is optimized for execution via **VS Code with Remote SSH connected to a GCP Compute Engine VM**.

## 🚀 Hybrid Workflow
1. **Frontend**: VS Code (Local) — Code editing, IntelliSense, Copilot, Remote SSH file explorer.
2. **Backend**: GCP Compute Engine VM (Debian 11, `e2-medium`) — Linux execution, EnergyPlus installation, GCS access.

## 📋 Prerequisites
- VS Code with [Google Cloud Code](https://marketplace.visualstudio.com/items?itemName=googlecloudtools.cloudcode) and [Google Compute Engine MCP Extension](https://marketplace.visualstudio.com/items?itemName=google.google-compute-engine-mcp-extension) installed.
- VM `sim-test-vm` provisioned and connected via Remote SSH.
- Python environment `.venv-test` set up on the VM (`bash config/setup_gcp_env.sh`).
- Access to GCS bucket `eplus-colab-cloud-data`.

## 🎯 Architecture
This notebook works directly with files stored in Google Cloud Storage:
- **IDF models** and **EPW weather files** are downloaded from the bucket
- **Simulations** run on the remote Compute Engine VM
- **Results** are automatically uploaded back to the bucket

No repository clone or GitHub tokens required. Authentication is handled by Workload Identity (Service Account attached to the VM).

---

## 🆕 Update: GCP VM Version

**Date**: May 2026

This notebook was adapted from the Colab VS Code version to run on a dedicated GCP Compute Engine VM.

### 🎯 Key Changes from Colab version:
- ✅ **Authentication**: Workload Identity (no interactive `gcloud auth` needed inside the VM)
- ✅ **Output prefix**: `results/` (consistent with all other project versions)
- ✅ **Output subfolder**: `gcp_vm_simulation_{timestamp}/`
- ✅ **Environment**: `.venv-test` isolated from the project's main `.venv`
- ✅ **File transfer**: Via VS Code Remote SSH file explorer (repo is private)

### 📝 How to Use:
1. Run cells in order (1 → 10)
2. Files are downloaded automatically from `models/` and `weather/`
3. Results are uploaded to `results/gcp_vm_simulation_{timestamp}/`
4. Use cells 11 and 12 to explore available files in the bucket

---

In [ ]:
# @title 1. GCP Authentication and Project Configuration
import os
import sys
import warnings
import time
from typing import Optional

# --- Configuration ---
PROJECT_ID = 'eplus-colab-cloud'  # @param {type:"string"}

def configure_gcp_project(project_id: str) -> None:
    """Configures the GCP project. On a Compute Engine VM with Workload Identity,
    ADC credentials are available automatically — no interactive login required."""
    print(f"🔐 Configuring project: {project_id}")
    os.environ['GOOGLE_CLOUD_PROJECT'] = project_id

    adc_path = os.path.expanduser("~/.config/gcloud/application_default_credentials.json")
    if os.path.exists(adc_path):
        print("✅ ADC credentials found (Workload Identity active).")
    else:
        print("⚠️  ADC credentials not found.")
        print("   On a GCP VM: credentials are provided automatically via Workload Identity.")
        print("   Locally: run 'gcloud auth application-default login'")

    try:
        import subprocess
        subprocess.run(
            ['gcloud', 'config', 'set', 'project', project_id],
            check=True, capture_output=True
        )
        print(f"✅ GCP project configured: {project_id}")
    except Exception as e:
        warnings.warn(f"Could not configure gcloud CLI (non-critical): {e}")

configure_gcp_project(PROJECT_ID)

In [ ]:
# @title 2. Resource Diagnostics (CPU/RAM)
import psutil
import os

print("--- Execution Environment Diagnostics ---")
try:
    cpu_count = os.cpu_count()
    print(f"🧠 Logical CPUs: {cpu_count}")

    ram_gb = psutil.virtual_memory().total / 1e9
    print(f"💾 Total RAM: {ram_gb:.2f} GB")

    if ram_gb < 20:
        print("⚠️  Warning: Standard RAM. Consider High-RAM for large models.")
    else:
        print("✅  High-RAM runtime detected.")

    gpu_info = !nvidia-smi
    gpu_info = '\n'.join(gpu_info)
    if 'failed' in gpu_info or 'not found' in gpu_info:
        print("ℹ️  No dedicated GPU detected (OK for EnergyPlus).")
    else:
        print("🚀 GPU detected (available for ML/TensorFlow).")
except Exception as e:
    print(f"Diagnostics error: {e}")

In [ ]:
# @title 3. Define GCS Bucket and Input Files
from google.cloud import storage
from typing import List

BUCKET_NAME = 'eplus-colab-cloud-data' # @param {type:"string"}

# Bucket structure:
# gs://eplus-colab-cloud-data/
#   ├── models/          ← IDF files
#   ├── weather/         ← EPW files
#   ├── results/         ← Simulation outputs
#   ├── scripts/         ← Installation scripts
#   └── notebooks/       ← Notebooks

# Input files (paths relative to bucket)
IDF_FILE = 'models/5ZoneAirCooled.idf'
EPW_FILE = 'weather/USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw'

print(f"🎯 Target Bucket: gs://{BUCKET_NAME}")
print(f"📁 IDF File: {IDF_FILE}")
print(f"🌤️  EPW File: {EPW_FILE}")

# Validate and inspect the bucket using Cloud Storage API
try:
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)

    if bucket.exists():
        print("✅ Bucket access confirmed.")

        print("\n📋 Bucket structure:")
        folders = ['models/', 'weather/']
        for folder in folders:
            print(f"\n📂 {folder}")
            blobs = list(client.list_blobs(BUCKET_NAME, prefix=folder, delimiter='/'))
            files = [blob.name for blob in blobs if not blob.name.endswith('/') and blob.name != folder]

            if files:
                for file_path in files[:5]:
                    filename = file_path.split('/')[-1]
                    blob = bucket.blob(file_path)
                    blob.reload()
                    size_mb = blob.size / (1024 * 1024) if blob.size else 0
                    print(f"  • {filename} ({size_mb:.2f} MB)")
                if len(files) > 5:
                    print(f"  ... and {len(files) - 5} more file(s)")
            else:
                print("  (empty)")

        print("\n🔍 Checking input files:")
        idf_blob = bucket.blob(IDF_FILE)
        epw_blob = bucket.blob(EPW_FILE)

        if idf_blob.exists():
            idf_blob.reload()
            print(f"  ✅ {IDF_FILE} ({idf_blob.size / (1024 * 1024):.2f} MB)")
        else:
            print(f"  ⚠️ {IDF_FILE} not found!")

        if epw_blob.exists():
            epw_blob.reload()
            print(f"  ✅ {EPW_FILE} ({epw_blob.size / (1024 * 1024):.2f} MB)")
        else:
            print(f"  ⚠️ {EPW_FILE} not found!")
    else:
        print(f"❌ Bucket '{BUCKET_NAME}' does not exist or is not accessible.")

except Exception as e:
    print(f"❌ Error accessing bucket: {e}")
    print("   Check that authentication (cell 1) ran correctly.")

## 🛠️ EnergyPlus v25.1.0 Installation
Installs the simulation engine on the remote Linux VM.

In [ ]:
# @title 4. Install EnergyPlus
from pathlib import Path
import subprocess

EPLUS_VERSION = "25.1.0"
EPLUS_URL = 'https://github.com/NREL/EnergyPlus/releases/download/v25.1.0/EnergyPlus-25.1.0-68a4a7c774-Linux-Ubuntu22.04-x86_64.run'
INSTALL_PATH = Path('/eplus')

def install_energyplus(url: str, install_path: Path) -> None:
    if install_path.exists():
        print(f"✅ EnergyPlus already installed at {install_path}")
        return

    print(f"⬇️ Downloading EnergyPlus v{EPLUS_VERSION}...")
    installer = Path('/tmp/ep_installer.run')
    subprocess.run(['wget', '-q', '-O', str(installer), url], check=True)
    subprocess.run(['chmod', '+x', str(installer)], check=True)

    print("📦 Installing system dependencies...")
    subprocess.run(['sudo', 'apt-get', '-qq', 'update'], check=True)
    deps = ['libxcb-icccm4', 'libxcb-image0', 'libxcb-keysyms1', 'libxcb-render-util0', 'libxcb-xinerama0', 'libxcb-xkb1', 'libxkbcommon-x11-0', 'libxcb-randr0', 'libxcb-shape0', 'libxcb-xfixes0', 'libxcb-sync1', 'libx11-xcb1', 'libxcb-cursor0', 'libsm6', 'libxext6', 'libxrender1', 'libfontconfig1', 'libfreetype6', 'libglib2.0-0', 'libdbus-1-3']
    subprocess.run(['sudo', 'apt-get', '-qq', 'install', '-y'] + deps, check=True)

    print("⚙️ Running installer...")
    # Create directory to avoid desktop entry error
    subprocess.run(['sudo', 'mkdir', '-p', '/root/.local/share/applications'], check=True)

    subprocess.run(['sudo', str(installer), 'install', '-c', '--al', '-t', str(install_path)], check=True)
    installer.unlink()
    print(f"✅ Installation complete at {install_path}")

install_energyplus(EPLUS_URL, INSTALL_PATH)

if str(INSTALL_PATH) not in sys.path:
    sys.path.insert(0, str(INSTALL_PATH))

In [ ]:
# @title 5. Configure Python API
import os
from pathlib import Path

try:
    install_path = Path(INSTALL_PATH)
    if not install_path.exists():
        raise FileNotFoundError(f"Installation directory not found: {install_path}")

    os.environ.setdefault("EPLUS_HOME", str(install_path))
    if str(install_path) not in sys.path:
        sys.path.insert(0, str(install_path))
        print(f"✅ EnergyPlus API linked to Python: {install_path}")
    else:
        print("ℹ️  API already in path.")

    from pyenergyplus.api import EnergyPlusAPI
    api = EnergyPlusAPI()
    print(f"✅ API loaded successfully! Engine version: {api.functional.ep_version()}")

except FileNotFoundError as e:
    print(f"❌ {e}")
    print("   Check that cell 4 (Install EnergyPlus) ran correctly.")
except Exception as e:
    print(f"❌ Fatal API error: {e}")

## ▶️ Running the Simulation
1. **Stage-In**: Download inputs from GCS to `/tmp`.
2. **Run**: Execute via `pyenergyplus`.
3. **Stage-Out**: Upload results back to GCS.

In [ ]:
# @title 6. Stage-In (Download Inputs)
from google.cloud import storage

WORK_DIR = Path('/tmp/energyplus_sim')
WORK_DIR.mkdir(exist_ok=True)

def download_inputs(bucket_name: str, files: dict[str, str], dest_dir: Path) -> dict[str, str]:
    """
    Download files from GCS to local directory.

    Args:
        bucket_name: GCS bucket name
        files: Dict {type: path_in_bucket} e.g. {'idf': 'models/file.idf'}
        dest_dir: Local destination directory

    Returns:
        Dict with local paths of downloaded files
    """
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    local_files = {}
    print(f"⬇️ Downloading files from gs://{bucket_name}...")

    for file_type, gcs_path in files.items():
        blob = bucket.blob(gcs_path)
        filename = Path(gcs_path).name
        dest = dest_dir / filename

        blob.download_to_filename(str(dest))
        local_files[file_type] = str(dest)
        print(f"  ✓ {gcs_path} → {filename}")

    return local_files

input_files = {
    'idf': IDF_FILE,
    'epw': EPW_FILE
}

local_paths = download_inputs(BUCKET_NAME, input_files, WORK_DIR)
local_idf = local_paths['idf']
local_epw = local_paths['epw']

print(f"\n📂 Local files ready:")
print(f"  IDF: {local_idf}")
print(f"  EPW: {local_epw}")

In [ ]:
# @title 7. Run Simulation (API)
from pyenergyplus.api import EnergyPlusAPI
api = EnergyPlusAPI()
state = api.state_manager.new_state()

args = [
    '-d', str(WORK_DIR),
    '-w', local_epw,
    local_idf
]

print(f"🚀 Starting simulation for {IDF_FILE}...")
exit_code = api.runtime.run_energyplus(state, args)

if exit_code == 0:
    print("\n✅ SIMULATION SUCCESSFUL!")
else:
    print(f"\n❌ Simulation failed. Exit code: {exit_code}")

api.state_manager.delete_state(state)

In [ ]:
# @title 8. Stage-Out (Upload Results)
from datetime import datetime

def upload_results(bucket_name: str, source_dir: Path, prefix: str) -> int:
    """
    Upload results to the GCS bucket.

    Args:
        bucket_name: GCS bucket name
        source_dir: Local directory containing results
        prefix: Bucket prefix (e.g. 'results/gcp_vm_simulation_20260206_120000')

    Returns:
        Number of files uploaded
    """
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    print(f"⬆️ Uploading results to gs://{bucket_name}/{prefix}...")
    count = 0

    for file_path in source_dir.iterdir():
        if file_path.is_file() and not file_path.name.endswith(('.idf', '.epw')):
            blob = bucket.blob(f"{prefix}/{file_path.name}")
            blob.upload_from_filename(str(file_path))
            print(f"  ✓ {file_path.name}")
            count += 1

    return count

if exit_code == 0:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    gcs_prefix = f"results/gcp_vm_simulation_{timestamp}"

    files_uploaded = upload_results(BUCKET_NAME, WORK_DIR, gcs_prefix)

    print(f"\n✅ {files_uploaded} result files uploaded successfully!")
    print(f"\n🔗 Results available at:")
    print(f"   GCS Browser: https://console.cloud.google.com/storage/browser/{BUCKET_NAME}/{gcs_prefix}")
    print(f"   gcloud: gcloud storage ls gs://{BUCKET_NAME}/{gcs_prefix}/")
else:
    print("⚠️ Simulation failed, upload cancelled.")

In [ ]:
# @title 9. View HTML Report
from IPython.display import HTML, display

report = WORK_DIR / 'eplustbl.htm'
if report.exists():
    with open(report, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read(5000)

    print("📄 Previewing report (truncated):")
    display(HTML(content + "...<br><br><b>⚠️ Full report available in the GCS bucket (link above).</b>"))
else:
    print("⚠️ HTML report not found.")

In [ ]:
# @title 10. Quick Visualization (Zone Temperatures)
import pandas as pd
import matplotlib.pyplot as plt

csv_path = WORK_DIR / 'eplusout.csv'

if csv_path.exists():
    print("📊 Generating temperature chart...")
    try:
        df = pd.read_csv(csv_path)
        temp_cols = [c for c in df.columns if 'Temperature' in c and 'Zone' in c]

        if temp_cols:
            plt.figure(figsize=(15, 6))
            for col in temp_cols[:5]:
                plt.plot(df[col], label=col)

            plt.title("Zone Temperature Profile")
            plt.xlabel("Time Step")
            plt.ylabel("Temperature (°C)")
            plt.legend(loc='best', fontsize='small')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print("ℹ️ No zone temperature columns found.")
    except Exception as e:
        print(f"❌ Plot error: {e}")
else:
    print("⚠️ Results file not found.")

In [ ]:
# @title 11. Utilities: Explore the Bucket
from google.cloud import storage
from typing import List

def list_bucket_structure(bucket_name: str) -> None:
    """Lists the bucket folder structure."""
    print(f"📁 Bucket structure gs://{bucket_name}:\n")
    folders = ['models/', 'weather/', 'results/', 'scripts/', 'notebooks/']
    for folder in folders:
        print(f"\n📂 {folder}")
        result = subprocess.run(
            ['gcloud', 'storage', 'ls', f'gs://{bucket_name}/{folder}'],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            files = [line.strip() for line in result.stdout.split('\n') if line.strip()]
            if files:
                for file in files[:10]:
                    filename = file.split('/')[-1]
                    if filename:
                        print(f"  • {filename}")
                if len(files) > 10:
                    print(f"  ... and {len(files) - 10} more files")
            else:
                print("  (empty)")
        else:
            print(f"  ⚠️ Could not list")

def list_available_models(bucket_name: str) -> List[str]:
    """Lists available IDF models."""
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)
    blobs = bucket.list_blobs(prefix='models/')
    models = [blob.name for blob in blobs if blob.name.endswith('.idf')]
    print("📋 Available IDF models:")
    for model in models:
        print(f"  • {model}")
    return models

def list_available_weather(bucket_name: str) -> List[str]:
    """Lists available EPW weather files."""
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)
    blobs = bucket.list_blobs(prefix='weather/')
    weather_files = [blob.name for blob in blobs if blob.name.endswith('.epw')]
    print("🌤️  Available weather files:")
    for wf in weather_files:
        print(f"  • {wf}")
    return weather_files

def list_recent_results(bucket_name: str, limit: int = 5) -> None:
    """Lists the most recent simulation results."""
    result = subprocess.run(
        ['gcloud', 'storage', 'ls', f'gs://{bucket_name}/results/'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        folders = [line.strip() for line in result.stdout.split('\n') if 'gcp_vm_simulation_' in line]
        folders.sort(reverse=True)
        print(f"📊 Last {min(limit, len(folders))} simulations:")
        for folder in folders[:limit]:
            sim_name = folder.rstrip('/').split('/')[-1]
            print(f"  • {sim_name}")
            print(f"    {folder}")
    else:
        print("⚠️ Could not list results")

print("💡 Available functions:")
print("  • list_bucket_structure(BUCKET_NAME)")
print("  • list_available_models(BUCKET_NAME)")
print("  • list_available_weather(BUCKET_NAME)")
print("  • list_recent_results(BUCKET_NAME)")
print("\nExample:")
print("  list_bucket_structure(BUCKET_NAME)")

## 📚 GCS Bucket Structure

The `eplus-colab-cloud-data` bucket is organized as follows:

```
gs://eplus-colab-cloud-data/
├── models/              # IDF files (building models)
│   └── 5ZoneAirCooled.idf
├── weather/             # EPW files (climate data)
│   └── USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw
├── results/             # Simulation outputs (organized by version + timestamp)
│   ├── gcp_vm_simulation_20260206_120000/
│   ├── colab_vs_code_simulation_20260206_110000/
│   └── cloud_shell_simulation_20260205_195217/
├── scripts/             # Installation and automation scripts
│   └── install_energyplus.sh
└── notebooks/           # Archived notebooks
    └── _legacy/
```

### 🔄 Workflow

1. **Models & Weather**: Stored in `models/` and `weather/`
2. **Execution**: Notebook downloads files, runs simulation on the VM
3. **Results**: Automatically uploaded to `results/gcp_vm_simulation_{timestamp}/`

### 📝 How to Add New Files

```bash
# Upload a new IDF model
gcloud storage cp my_model.idf gs://eplus-colab-cloud-data/models/

# Upload a weather file
gcloud storage cp city.epw gs://eplus-colab-cloud-data/weather/

# List results
gcloud storage ls gs://eplus-colab-cloud-data/results/
```

In [ ]:
# @title 12. [EXAMPLE] Explore Available Files
# Run this cell to see all available files in the bucket

print("="*60)
print("🔍 EXPLORING GCS BUCKET")
print("="*60)

print("\n")
models = list_available_models(BUCKET_NAME)

print("\n")
weather = list_available_weather(BUCKET_NAME)

print("\n")
list_recent_results(BUCKET_NAME, limit=10)

print("\n" + "="*60)
print(f"✅ Total: {len(models)} models, {len(weather)} weather files")
print("="*60)

## 📚 Guide: Adding New Files to the Bucket

### 🎯 Goal
Add new IDF models and EPW weather files to the GCS bucket for use in future simulations.

### 📋 Prerequisites
- `gcloud` CLI installed and configured
- GCP project authenticated (done in Cell 1)
- Access to `gs://eplus-colab-cloud-data`

---

## 🔧 Method 1: Upload via gcloud storage (Recommended)

### IDF Models:
```bash
# Upload a single model
gcloud storage cp my_model.idf gs://eplus-colab-cloud-data/models/

# Upload multiple models
gcloud storage cp /local/path/models/*.idf gs://eplus-colab-cloud-data/models/
```

### EPW Weather Files:
```bash
# Upload a single weather file
gcloud storage cp chicago.epw gs://eplus-colab-cloud-data/weather/

# Upload multiple files
gcloud storage cp /local/path/weather/*.epw gs://eplus-colab-cloud-data/weather/
```

> **Note:** Use `gcloud storage` instead of `gsutil` — up to 94% faster for large IDF/EPW volumes.

---

## 🐍 Method 2: Upload via Python (in notebook)

```python
from google.cloud import storage
from pathlib import Path

def upload_file_to_gcs(local_path: str, gcs_folder: str):
    """Upload a file to the GCS bucket."""
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)

    local_file = Path(local_path)
    if not local_file.exists():
        print(f"❌ File not found: {local_path}")
        return

    gcs_path = f"{gcs_folder.rstrip('/')}/{local_file.name}"
    blob = bucket.blob(gcs_path)

    print(f"⬆️ Uploading {local_file.name}...")
    blob.upload_from_filename(str(local_file))
    print(f"✅ Uploaded to gs://{BUCKET_NAME}/{gcs_path}")

# Examples:
# upload_file_to_gcs("my_model.idf", "models")
# upload_file_to_gcs("chicago.epw", "weather")
```

---

## 📂 Expected Bucket Structure

```
gs://eplus-colab-cloud-data/
├── models/
│   ├── 5ZoneAirCooled.idf
│   ├── Office_Medium.idf
│   └── Hospital_Large.idf
├── weather/
│   ├── USA_IL_Chicago-OHare...epw
│   ├── USA_CA_Los_Angeles...epw
│   └── BRA_RJ_Rio_de_Janeiro...epw
├── results/
│   ├── gcp_vm_simulation_20260206_211001/
│   └── colab_vs_code_simulation_20260206_180000/
├── scripts/
│   └── install_energyplus.sh
└── notebooks/
    └── EnergyPlus_API_GCP_VM.ipynb
```

---

## 🔍 Verify Uploads

```bash
# List models
gcloud storage ls gs://eplus-colab-cloud-data/models/

# List weather files
gcloud storage ls gs://eplus-colab-cloud-data/weather/

# Check total bucket size
gcloud storage du --summarize gs://eplus-colab-cloud-data/
```

---

## 🔄 Workflow for New Simulations

1. Upload IDF and EPW files to the bucket
2. Update Cell 3 of the notebook:
```python
IDF_FILE = 'models/my_new_model.idf'
EPW_FILE = 'weather/new_city.epw'
```
3. Run cells 6 → 7 → 8 (Stage-In → Simulation → Stage-Out)
4. Results will be uploaded to `results/gcp_vm_simulation_{timestamp}/`

---

## 📞 Troubleshooting

| Problem | Solution |
|---------|----------|
| `gcloud: command not found` | Install Google Cloud SDK: `curl https://sdk.cloud.google.com \| bash` |
| `AccessDenied` on upload | Check IAM permissions or Workload Identity configuration |
| File not listed after upload | Wait 1-2 minutes (listing cache) or use `gcloud storage ls --stat` |
| Slow upload | `gcloud storage` handles parallelism automatically |